## Setting Up the Project Environment - Formula 1 Data Project

### What are we doing here?
Before we can analyze Formula 1 race data, we need to **prepare the workspace** - just like setting up a desk, folders, and drawers before starting office work.

Our raw data (CSV files with race results, driver info, circuits, etc.) lives in **Azure Data Lake** (cloud storage). This notebook connects Databricks to that storage and organizes everything neatly.

---

### Why do we need each of these?

| What we create | Why we need it | Real-world analogy |
| --- | --- | --- |
| **External Location** | Gives Databricks *permission* to access files in Azure | Like giving someone a key to your storage room |
| **Catalog** | A top-level container to group all project data | Like a filing cabinet labeled "Formula 1" |
| **Schemas** (landing, bronze, silver, gold) | Separates data by quality/stage of processing | Like drawers in the cabinet - one for raw papers, one for reviewed, one for final reports |
| **Volume** | Lets us browse and read raw files easily inside Databricks | Like mounting a USB drive so you can open files directly |

---

### Steps in this notebook:
1. **Verify cloud access** - Confirm we can see our files in Azure
2. **Create External Location** - Grant Databricks permission to read/write in Azure
3. **Create Catalog** - Set up the main container for all Formula 1 data
4. **Create Schemas** - Organize data into layers (landing > bronze > silver > gold)
5. **Create Volume** - Make raw files accessible via a simple file path

#### Verify Access to Cloud Storage
Let's first check that we can see our data files in Azure Data Lake. The command below lists the contents of the `landing` folder - this is where our raw CSV files are stored.

In [0]:
%fs ls 'abfss://formula1@databricksprojectsextdl1.dfs.core.windows.net/landing'

#### Create an External Location
An **external location** is like a secure bridge between Databricks and your Azure storage account. It tells Databricks: *"You are allowed to read and write data at this specific cloud address."* Without it, Databricks cannot access your files in Azure.

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS databricks_projects_ext_dl1_formula1
    URL 'abfss://formula1@databricksprojectsextdl1.dfs.core.windows.net/'
    WITH (STORAGE CREDENTIAL `databricks-course-sc`)
    COMMENT 'External location for databricks projects';

### Create a Unity Catalog for the Project
A **catalog** is the highest-level container in Databricks for organizing your data. Think of it like a filing cabinet - inside it, you'll have drawers (schemas) that hold your actual tables and files. We're creating one called `formula1` to keep all our project data together.

##### View Existing Catalogs
Before creating a new catalog, let's see what catalogs already exist in our workspace. This helps confirm our environment and avoid duplicates.

In [0]:
%sql
SHOW CATALOGS

##### Create the Formula 1 Catalog
Now we create our project catalog called `formula1`. All tables, schemas, and volumes for this project will live inside it. The `MANAGED LOCATION` tells Databricks where to physically store the data in Azure.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS  formula1
   MANAGED LOCATION 'abfss://formula1@databricksprojectsextdl1.dfs.core.windows.net/' 
   COMMENT 'This is the main catalog for our Formula 1 project';

#### Create Schemas (Databases) for Each Data Layer
Inside our catalog, we create separate **schemas** (also called databases) to organize data by quality level:
- **landing** - Raw files as they arrive (untouched)
- **bronze** - Raw data loaded into tables (minimal processing)
- **silver** - Cleaned and transformed data (ready for analysis)
- **gold** - Business-level aggregations (dashboards, reports)

This layered approach is called the **Medallion Architecture** - data gets progressively cleaner as it moves through each layer.

In [0]:
%sql
CREATE  SCHEMA  IF NOT EXISTS formula1.landing;
CREATE  SCHEMA  IF NOT EXISTS formula1.bronze
     MANAGED LOCATION 'abfss://formula1@databricksprojectsextdl1.dfs.core.windows.net/bronze';
CREATE  SCHEMA  IF NOT EXISTS formula1.silver
     MANAGED LOCATION 'abfss://formula1@databricksprojectsextdl1.dfs.core.windows.net/silver';
CREATE  SCHEMA  IF NOT EXISTS formula1.gold
     MANAGED LOCATION 'abfss://formula1@databricksprojectsextdl1.dfs.core.windows.net/gold';
CREATE  SCHEMA  IF NOT EXISTS formula1.practice_class
     MANAGED LOCATION 'abfss://formula1@databricksprojectsextdl1.dfs.core.windows.net/practice_class';
CREATE  SCHEMA  IF NOT EXISTS formula1.practice_class_silver
     MANAGED LOCATION 'abfss://formula1@databricksprojectsextdl1.dfs.core.windows.net/practice_class_silver'

#### Verify the Schemas Were Created
Let's confirm that all four schemas (landing, bronze, silver, gold) now exist inside our `formula1` catalog.

In [0]:
%sql
use catalog formula1;
show schemas;

#### Create a Volume for Raw Files
A **volume** lets you access files (like CSVs, JSONs, etc.) directly from Databricks - similar to a shared folder on your computer. We'll create one that points to our `landing` folder in Azure, so we can easily read the raw Formula 1 data files.

> **Quick note: External Location vs Volume**
>
> | Concept | Purpose |
> | --- | --- |
> | **External Location** | Grants Databricks *permission* to access a cloud storage path securely |
> | **Volume** | Gives you a *file-system-like interface* to browse and read those files inside Databricks |
>
> You need the external location first (the permission), then the volume (the easy access).

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS formula1.landing.files
LOCATION 'abfss://formula1@databricksprojectsextdl1.dfs.core.windows.net/landing';

In [0]:
%fs ls /Volumes/formula1/landing/files